<a href="https://colab.research.google.com/github/bvsvntv/ai-ml-learning/blob/main/notebooks/lang-graph/subgraph_with_separate_states.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.1/587.1 kB 6.3 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [langchain-openai]


In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from google.colab import userdata

In [4]:
api_key = userdata.get("OPENAI_API_KEY")

In [13]:
class ParentState(TypedDict):
  question: str
  answer_eng: str
  answer_nep: str

In [7]:
class SubState(TypedDict):
  input_text: str
  translated_text: str

In [9]:
llm_1 = ChatOpenAI(api_key=api_key, model="gpt-6-luna")
llm_2 = ChatOpenAI(api_key=api_key, model="gpt-6-luna")

### SubGraph

In [10]:
def translate_text(state: SubState):
  prompt = f"""
  Translate the following text to Nepali.
  Keep it natural and clear. Do not add extra content.

  Text:
  {state["input_text"]}
  """.strip()

  translated_text = llm_2.invoke(prompt).content
  return {"translated_text": translated_text}

In [20]:
subgraph = StateGraph(SubState)

subgraph.add_node("translate_text", translate_text)

subgraph.add_edge(START, "translate_text")
subgraph.add_edge("translate_text", END)

sub_workflow = subgraph.compile()

### ParentGraph

In [21]:
def generate_answer(state: ParentState):
  prompt = f"You are a helpful assistant. Answer the following question: {state["question"]}"

  answer = llm_1.invoke(prompt).content
  return {"answer_eng": answer}

In [22]:
def translate_answer(state: ParentState):
  # call subgraph
  result = sub_workflow.invoke({"input_text": state["answer_eng"]})

  return {"answer_nep": result["translated_text"]}

In [23]:
graph = StateGraph(ParentState)

graph.add_node("answer", generate_answer)
graph.add_node("translate", translate_answer)

graph.add_edge(START, "answer")
graph.add_edge("answer", "translate")
graph.add_edge("translate", END)

parent_workflow = graph.compile()

In [25]:
initial_state = {
    "question": "What is photosynthesis?"
}
result = parent_workflow.invoke(initial_state)

print(result["answer_eng"])
print()
print(result["answer_nep"])

Photosynthesis is the process by which plants, algae, and some bacteria use sunlight to convert carbon dioxide and water into sugars, storing energy in the sugars and releasing oxygen.

A simplified equation is:

**6 CO₂ + 6 H₂O + light → C₆H₁₂O₆ + 6 O₂**

प्रकाश संश्लेषण त्यो प्रक्रिया हो जसद्वारा बिरुवा, शैवाल र केही ब्याक्टेरियाले सूर्यको प्रकाश प्रयोग गरेर कार्बन डाइअक्साइड र पानीलाई शर्करामा रूपान्तरण गर्छन्। यस क्रममा शर्करामा ऊर्जा सञ्चित हुन्छ र अक्सिजन निस्कन्छ।

सरलीकृत समीकरण:

**6 CO₂ + 6 H₂O + प्रकाश → C₆H₁₂O₆ + 6 O₂**
